# Summer Storm Langlois Size Class HI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os 
from scipy.optimize import curve_fit
from scipy.integrate import quad

storm_directory = 'pulse_experiments'
storms = {}
for filename in os.listdir(storm_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(storm_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]  # drop stray unnamed columns
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        storms[key] = df                                      # store the data frame in the dictionary


KeyError: ['Date_Time']

In [12]:
for storm_name, storm_df in storms.items():
    merged_storm_df = storm_df.join(shear_stress, how="left")
    storms[storm_name] = merged_storm_df
storms['st5_down']

,SSC (mg/L),Clay SSC (mg/L),Silt SSC (mg/L),Fine sand SSC (mg/L),shear_stress
Date_Time,,,,,
2023-08-13 18:30:00,13.33,0.184661,8.184764,4.960576,63.445700
2023-08-13 18:45:00,76.15,1.701191,47.584612,26.865720,66.830985
2023-08-13 18:58:00,41.20,1.008576,27.301592,12.889008,67.974764
2023-08-13 19:02:00,39.60,0.806652,24.868800,13.924944,67.931246
2023-08-13 19:10:00,27.60,0.586224,17.647716,9.367164,67.053310
2023-08-13 21:02:00,20.83,0.522000,13.767380,6.540412,68.396623


Hysteresis index calculation functions

In [13]:
## regression equations
# linear
def linear_func(Q, a, b):
    return a * Q + b
# logarithmic
def log_func(Q, a, b):
    return a * np.log(Q) + b
# exponential
def exp_func(Q, a, b):
    return a * np.exp(b * Q)

# split hydrograph into rising and falling limbs based on peak flow
def split_hydrograph(df, q_col):
    # if df empty or q_col has no valid values, return empty limbs
    if df.empty or df[q_col].dropna().empty:
        return df.copy(), df.copy()
    peak_time = df[q_col].idxmax()
    rising = df.loc[:peak_time].copy()
    falling = df.loc[peak_time:].copy()
    return rising, falling

# fit curves and calculate R²
def fit_best_curve(x, y):
    candidate_functions = {
        'linear': (linear_func, [1, 1]),
        'log': (log_func, [1, 1]),
        'exponential': (exp_func, [1, -0.01])
    }
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    mask = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x[mask])
    y = np.asarray(y[mask])

    # minimum points
    if len(x) < 2:
        return None
    best_r2 = -np.inf
    best_result = None
    # try each function and keep the one with the best r2
    for func_name, (func, p0) in candidate_functions.items():
        try:
            # avoid invalid log fits
            if func_name == 'log' and np.any(x <= 0):
                continue
            popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
            y_pred = func(x, *popt) # predicted values
            # residuals
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            # avoid divide-by-zero
            if np.isclose(ss_tot, 0):
                r2 = np.nan
            else:
                r2 = 1 - (ss_res / ss_tot)
            # keep best fit
            if np.isfinite(r2) and r2 > best_r2:
                best_r2 = r2
                best_result = {
                    'function_name': func_name,
                    'function': func,
                    'params': popt,
                    'r2': r2
                }
        except Exception:
            continue
    return best_result

# Langlois 2025 H calculation
def compute_langlois_H(event_df, tau_col, constituent_col, r2_threshold=0.50, storm_name=None):
    # data cleanup
    df = event_df[[tau_col, constituent_col]].dropna()
    if df.empty or df[tau_col].dropna().empty or df[constituent_col].dropna().empty:
        print(f"No valid {tau_col} or {constituent_col} for {storm_name or 'unknown'}; skipping")
        return None

    rising, falling = split_hydrograph(df, tau_col)
    if rising.empty or falling.empty:
        print(f"No rising or falling limb for {storm_name or 'unknown'}; skipping")
        return None

    # shear stress overlap range
    tau_min = max(rising[tau_col].min(), falling[tau_col].min())
    tau_max = min(rising[tau_col].max(), falling[tau_col].max())

    # fit rising limb
    rise_fit = fit_best_curve(rising[tau_col].values, rising[constituent_col].values)
    if rise_fit is None:
        print(f"Could not fit rising limb in " f"{storm_name or 'unknown'} for {constituent_col}")
        return None
    # fit falling limb
    fall_fit = fit_best_curve(falling[tau_col].values, falling[constituent_col].values)
    if fall_fit is None:
        print(f"Could not fit falling limb in " f"{storm_name or 'unknown'} for {constituent_col}")
        return None

    # check fit quality with r2 threshold
    if (rise_fit['r2'] < r2_threshold) or (fall_fit['r2'] < r2_threshold):
            label = storm_name if storm_name is not None else "unknown storm"
            print(f"Poor fit for rising (R²={rise_fit['r2']:.2f}) or falling (R²={fall_fit['r2']:.2f}) limb in {label} for {constituent_col}")
    # integrated areas
    rise_area, _ = quad(lambda q: rise_fit["function"](q, *rise_fit["params"]), tau_min, tau_max)
    fall_area, _ = quad(lambda q: fall_fit["function"](q, *fall_fit["params"]), tau_min, tau_max)
    # hysteresis index
    H = rise_area / fall_area

    return {
        # hysteresis
        'H': H,
        # rising limb
        'rise_r2': rise_fit['r2'],
        'rise_function': rise_fit['function'],
        'rise_params': rise_fit['params'],
        'rise_area': rise_area,
        # falling limb
        'fall_r2': fall_fit['r2'],
        'fall_function': fall_fit['function'],
        'fall_params': fall_fit['params'],
        'fall_area': fall_area,
        # overlap range
        'tau_min': tau_min,
        'tau_max': tau_max,
        # point counts
        'n_rising': len(rising),
        'n_falling': len(falling)
    }

Calculate H for all events

In [17]:
all_results = []

for storm_name, storm_df in storms.items():
    for constituent in ["SSC (mg/L)", "Clay SSC (mg/L)", "Silt SSC (mg/L)", "Fine sand SSC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue

        result = compute_langlois_H(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('summer_HI/langlois_hysteresis_summer.csv', index=False)

Poor fit for rising (R²=0.36) or falling (R²=0.85) limb in st1_down for SSC (mg/L)
Poor fit for rising (R²=0.36) or falling (R²=0.96) limb in st1_down for Clay SSC (mg/L)
Poor fit for rising (R²=0.35) or falling (R²=0.79) limb in st1_down for Silt SSC (mg/L)
Poor fit for rising (R²=0.33) or falling (R²=0.60) limb in st1_down for Fine sand SSC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.94) limb in st1_up for SSC (mg/L)
Poor fit for rising (R²=0.43) or falling (R²=0.99) limb in st1_up for Silt SSC (mg/L)
Poor fit for rising (R²=0.40) or falling (R²=0.99) limb in st2_down for SSC (mg/L)
Poor fit for rising (R²=0.20) or falling (R²=1.00) limb in st2_down for Fine sand SSC (mg/L)
Poor fit for rising (R²=0.16) or falling (R²=0.03) limb in st4_down for SSC (mg/L)
Poor fit for rising (R²=0.76) or falling (R²=0.39) limb in st4_down for Clay SSC (mg/L)
Poor fit for rising (R²=0.68) or falling (R²=0.39) limb in st4_down for Silt SSC (mg/L)
Poor fit for rising (R²=0.36) or falling (R²=0

C:\Users\nicol\AppData\Local\Temp\ipykernel_31864\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Poor fit for rising (R²=0.49) or falling (R²=0.59) limb in st7_up for SSC (mg/L)
Poor fit for rising (R²=0.65) or falling (R²=0.43) limb in st7_up for Clay SSC (mg/L)
Poor fit for rising (R²=0.59) or falling (R²=0.42) limb in st7_up for Silt SSC (mg/L)
Poor fit for rising (R²=0.18) or falling (R²=0.83) limb in st7_up for Fine sand SSC (mg/L)


### Plots

In [18]:
def plot_langlois_hysteresis(event_df, tau_col, constituent_col, r2_threshold=0.5, storm_name=None, 
                            out_dir='plots', save=True, show=False):

    df = event_df[[tau_col, constituent_col]].dropna()
    if df.empty:
        return None
    rising, falling = split_hydrograph(df, tau_col)
    if rising.empty or falling.empty:
        return None
    
    # reuse the same fitting logic as the H calculation
    result = compute_langlois_H(
        event_df,
        tau_col=tau_col,
        constituent_col=constituent_col,
        r2_threshold=r2_threshold,
        storm_name=storm_name,
    )
    if result is None:
        return None

    tau_min = result["tau_min"]
    tau_max = result["tau_max"]
    if not np.isfinite(tau_min) or not np.isfinite(tau_max) or tau_min >= tau_max:
        return None

    tau_fit = np.linspace(tau_min, tau_max, 200)

    rise_params = result["rise_params"]
    fall_params = result["fall_params"]
    rise_r2 = result["rise_r2"]
    fall_r2 = result["fall_r2"]
    H = result["H"]

    rise_fit = result["rise_function"](tau_fit, *result["rise_params"])
    fall_fit = result["fall_function"](tau_fit, *result["fall_params"])

    # PLOT 
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # time series
    ax = axes[0]
    ax.plot(df.index, df[tau_col], color="tab:blue", linewidth=1.5, label=tau_col)
    ax.set_ylabel(tau_col, color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.xaxis.set_major_locator(plt.MaxNLocator(8))
    ax2 = ax.twinx()
    ax2.plot(df.index, df[constituent_col], color="tab:red", linewidth=1.5, label=constituent_col)
    ax2.set_ylabel(constituent_col, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    ax.set_title("Event time series")

    # hysteresis loop
    ax = axes[1]
    ax.scatter(rising[tau_col], rising[constituent_col], label='Rising limb', color='tab:orange')
    ax.scatter(falling[tau_col], falling[constituent_col], label='Falling limb', color='tab:green')
    ax.plot(tau_fit, rise_fit, linewidth=2, label=f'Rising fit (R²={rise_r2:.2f})', color='tab:orange')
    ax.plot(tau_fit, fall_fit, linewidth=2, label=f'Falling fit (R²={fall_r2:.2f})', color='tab:green')

    ax.set_xlabel(tau_col)
    ax.set_ylabel(constituent_col)
    ax.set_title(f'H = {H:.2f}')
    ax.legend()

    # add a main title for the whole figure
    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    plt.suptitle(main_title, fontsize=15)
    plt.tight_layout()

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = f"{storm_name}_{constituent_col}_langlois.png".replace(" ", "_").replace("/", "_")
        fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    plt.close(fig)
    return result

In [21]:
all_results = []

for storm_name, storm_df in storms.items():
    for constituent in ["SSC (mg/L)", "Clay SSC (mg/L)", "Silt SSC (mg/L)", "Fine sand SSC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue

        plot_langlois_hysteresis(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name,
            out_dir='plots/langlois',)

Poor fit for rising (R²=0.36) or falling (R²=0.85) limb in st1_down for SSC (mg/L)
Poor fit for rising (R²=0.36) or falling (R²=0.96) limb in st1_down for Clay SSC (mg/L)
Poor fit for rising (R²=0.35) or falling (R²=0.79) limb in st1_down for Silt SSC (mg/L)
Poor fit for rising (R²=0.33) or falling (R²=0.60) limb in st1_down for Fine sand SSC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.94) limb in st1_up for SSC (mg/L)
Poor fit for rising (R²=0.43) or falling (R²=0.99) limb in st1_up for Silt SSC (mg/L)
Poor fit for rising (R²=0.40) or falling (R²=0.99) limb in st2_down for SSC (mg/L)


C:\Users\nicol\AppData\Local\Temp\ipykernel_31864\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Poor fit for rising (R²=0.20) or falling (R²=1.00) limb in st2_down for Fine sand SSC (mg/L)
Poor fit for rising (R²=0.16) or falling (R²=0.03) limb in st4_down for SSC (mg/L)
Poor fit for rising (R²=0.76) or falling (R²=0.39) limb in st4_down for Clay SSC (mg/L)
Poor fit for rising (R²=0.68) or falling (R²=0.39) limb in st4_down for Silt SSC (mg/L)
Poor fit for rising (R²=0.36) or falling (R²=0.52) limb in st4_up for SSC (mg/L)
Poor fit for rising (R²=0.45) or falling (R²=0.62) limb in st4_up for Silt SSC (mg/L)
Poor fit for rising (R²=0.18) or falling (R²=0.20) limb in st4_up for Fine sand SSC (mg/L)
Could not fit falling limb in st5_down for SSC (mg/L)
Could not fit falling limb in st5_down for Clay SSC (mg/L)
Could not fit falling limb in st5_down for Silt SSC (mg/L)
Could not fit falling limb in st5_down for Fine sand SSC (mg/L)
Poor fit for rising (R²=0.20) or falling (R²=0.01) limb in st5_up for SSC (mg/L)
Poor fit for rising (R²=0.03) or falling (R²=0.20) limb in st5_up for Cla

# Spring Event Langlois HI 

In [24]:
event_directory = 'size_concentrations/spring_events'
events = {}
for filename in os.listdir(event_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(event_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')] 
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        events[key] = df                                    # store the data frame in the dictionary

shear_stress = pd.read_csv('../../data/shear_stress/average_total_shear_stress_corrected.csv', parse_dates=['datetime'], index_col='datetime')
shear_stress = shear_stress.resample('1min').interpolate()

In [25]:
for event_name, event_df in events.items():
    merged_event_df = event_df.join(shear_stress, how="left")
    events[event_name] = merged_event_df

events['up_event14']

,SSC (mg/L),Clay SSC (mg/L),Silt SSC (mg/L),Fine sand SSC (mg/L),shear_stress
Date_Time,,,,,
2023-05-04 11:00:00,NaN,NaN,NaN,NaN,117.599485
2023-05-04 14:00:00,1.290323,NaN,NaN,NaN,121.280231
2023-05-04 17:00:00,NaN,NaN,NaN,NaN,123.582933
2023-05-04 20:00:00,0.714286,0.018767,0.563223,0.132296,125.705675
2023-05-04 23:00:00,5.714286,0.124695,4.063751,1.525840,124.885281
2023-05-05 02:00:00,NaN,NaN,NaN,NaN,123.485743
2023-05-05 05:00:00,2.142857,NaN,NaN,NaN,121.603735
2023-05-05 08:00:00,NaN,NaN,NaN,NaN,121.577074
2023-05-05 11:00:00,0.666667,0.015939,0.457360,0.193367,118.645647


Calculate HI for all events

In [26]:
all_results = []

for event_name, event_df in events.items():
    for constituent in ["SSC (mg/L)", "Clay SSC (mg/L)", "Silt SSC (mg/L)", "Fine sand SSC (mg/L)"]:
        if constituent not in event_df.columns:
            continue

        result = compute_langlois_H(
            event_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=event_name)

        if result is not None:
            result["event"] = event_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('spring_HI/langlois_hysteresis_spring.csv', index=False)

C:\Users\nicol\AppData\Local\Temp\ipykernel_31864\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Could not fit rising limb in down_event10 for Clay SSC (mg/L)
Could not fit rising limb in down_event10 for Silt SSC (mg/L)
Could not fit rising limb in down_event10 for Fine sand SSC (mg/L)
Could not fit falling limb in down_event11 for SSC (mg/L)
Could not fit falling limb in down_event11 for Clay SSC (mg/L)
Could not fit falling limb in down_event11 for Silt SSC (mg/L)
Could not fit falling limb in down_event11 for Fine sand SSC (mg/L)
Could not fit rising limb in down_event12 for SSC (mg/L)
Could not fit rising limb in down_event12 for Clay SSC (mg/L)
Could not fit rising limb in down_event12 for Silt SSC (mg/L)
Could not fit rising limb in down_event12 for Fine sand SSC (mg/L)
Could not fit rising limb in down_event13 for Clay SSC (mg/L)
Could not fit rising limb in down_event13 for Silt SSC (mg/L)
Could not fit rising limb in down_event13 for Fine sand SSC (mg/L)
Could not fit rising limb in down_event14 for SSC (mg/L)
Could not fit rising limb in down_event14 for Clay SSC (mg/L)

In [27]:
all_results = []

for event_name, event_df in events.items():
    for constituent in ["SSC (mg/L)", "Clay SSC (mg/L)", "Silt SSC (mg/L)", "Fine sand SSC (mg/L)"]:
        if constituent not in event_df.columns:
            continue

        plot_langlois_hysteresis(
            event_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=event_name,
            out_dir='plots/langlois')

C:\Users\nicol\AppData\Local\Temp\ipykernel_31864\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Could not fit rising limb in down_event10 for Clay SSC (mg/L)
Could not fit rising limb in down_event10 for Silt SSC (mg/L)
Could not fit rising limb in down_event10 for Fine sand SSC (mg/L)
Could not fit falling limb in down_event11 for SSC (mg/L)
Could not fit falling limb in down_event11 for Clay SSC (mg/L)
Could not fit falling limb in down_event11 for Silt SSC (mg/L)
Could not fit falling limb in down_event11 for Fine sand SSC (mg/L)
Could not fit rising limb in down_event12 for SSC (mg/L)
Could not fit rising limb in down_event12 for Clay SSC (mg/L)
Could not fit rising limb in down_event12 for Silt SSC (mg/L)
Could not fit rising limb in down_event12 for Fine sand SSC (mg/L)
Could not fit rising limb in down_event13 for Clay SSC (mg/L)
Could not fit rising limb in down_event13 for Silt SSC (mg/L)
Could not fit rising limb in down_event13 for Fine sand SSC (mg/L)
Could not fit rising limb in down_event14 for SSC (mg/L)
Could not fit rising limb in down_event14 for Clay SSC (mg/L)